# Tutorial 5 — Real-Time Training Dashboards

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part II — Debugging & Observability**  
**Follows:** Tutorial 4 (Gradient Hooks & Distribution Monitoring)  
**Precedes:** Tutorial 6 (Diagnosing Training Pathologies)

---

## What This Tutorial Covers

A training run that cannot be observed is difficult to debug. The hooks from Tutorial 4 provide the data; this tutorial provides the display. This tutorial presents two complementary dashboards:

- A **live Matplotlib dashboard** that updates in your terminal every N steps — no external services, no internet, works on a remote GPU box over SSH
- A **TensorBoard integration** that records the full history to disk and lets you compare multiple runs side by side after the fact

And one structured logging layer underneath both — a `TrainingLogger` that writes every metric to a JSONL file so your data is never locked into a visualization tool.

What to log, and why:

- **Train and eval loss** — the primary signal; everything else is diagnostic
- **Global gradient norm** — spikes here precede loss spikes; the early warning system
- **Per-layer gradient-to-weight ratio** — from Tutorial 4; tells you which layer is misbehaving
- **Weight norms** — slow drift here signals that a layer is being over-updated or regularized into zero
- **Learning rate** — critical during warmup and decay phases; easy to forget to log
- **Tokens per second** — throughput; tells you if a code change accidentally made training slower
- **GPU memory** — catches allocation leaks before they OOM-kill your run

The sawtooth chart — what it is, what causes it, and what it tells you about your training dynamics — is covered in Section 5.

---

## 1. What to Log and Why

Before building anything, be precise about what each metric tells you and when you should look at it.

```
Metric                  Frequency   What it tells you
─────────────────────────────────────────────────────────────────────────
train_loss              every step  Primary learning signal
eval_loss               every ~500  Generalization; divergence from train = overfit
global_grad_norm        every step  Spikes = instability; log pre-clip norm
grad_weight_ratio       every ~50   Per-layer health; use LightweightMonitor
weight_norm             every ~50   Slow growth = under-regularized; decay = dying
learning_rate           every step  Essential during warmup/decay phases
tokens_per_sec          every step  Throughput regression detector
gpu_memory_gb           every ~50   Leak detector
```

[Log more than you think you need.]{.underline} Disk is cheap; rerunning a 12-hour training job to get a metric you forgot to record is not.

---

## 2. The `TrainingLogger` — Structured Logging First

Before any visualization, write everything to disk in a structured format. [Visualization tools come and go; a JSONL file is forever.]{.mark}

In [ ]:
import json
import time
import os
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

@dataclass
class StepRecord:
    """One record per training step."""
    step:              int
    timestamp:         float
    train_loss:        float
    learning_rate:     float
    global_grad_norm:  float
    tokens_per_sec:    float
    # Optional — logged less frequently
    eval_loss:         Optional[float] = None
    mean_grad_ratio:   Optional[float] = None
    min_grad_ratio:    Optional[float] = None
    max_grad_ratio:    Optional[float] = None
    mean_weight_norm:  Optional[float] = None
    gpu_memory_gb:     Optional[float] = None


class TrainingLogger:
    """
    Writes training metrics to a JSONL file (one JSON object per line).
    Optionally feeds TensorBoard and the live dashboard.

    Usage:
        logger = TrainingLogger(run_dir='runs/my_experiment')

        for step in range(max_steps):
            # ... training step ...
            logger.log_step(
                step=step,
                train_loss=loss.item(),
                learning_rate=scheduler.get_last_lr()[0],
                global_grad_norm=grad_norm,
                tokens_per_sec=tps,
            )

            if step % 500 == 0:
                logger.log_eval(step, eval_loss)
    """

    def __init__(
        self,
        run_dir: str,
        run_name: str = None,
        use_tensorboard: bool = False,
    ):
        self.run_dir  = Path(run_dir)
        self.run_name = run_name or f"run_{int(time.time())}"
        self.run_dir.mkdir(parents=True, exist_ok=True)

        self.log_path = self.run_dir / 'metrics.jsonl'
        self._log_file = open(self.log_path, 'a')   # append — survives restarts

        self._step_start_time = time.time()
        self._step_start_tokens = 0
        self._records: list[StepRecord] = []

        # TensorBoard writer (optional)
        self._tb_writer = None
        if use_tensorboard:
            try:
                from torch.utils.tensorboard import SummaryWriter
                tb_dir = self.run_dir / 'tensorboard' / self.run_name
                self._tb_writer = SummaryWriter(str(tb_dir))
                print(f"TensorBoard logging to: {tb_dir}")
                print(f"  Launch with: tensorboard --logdir {self.run_dir / 'tensorboard'}")
            except ImportError:
                print("TensorBoard not available — pip install tensorboard")

        print(f"TrainingLogger initialized. Metrics → {self.log_path}")

    # ------------------------------------------------------------------ #
    #  Timing helpers                                                      #
    # ------------------------------------------------------------------ #

    def start_step(self, tokens_so_far: int):
        """Call at the start of each training step for throughput tracking."""
        self._step_start_time   = time.time()
        self._step_start_tokens = tokens_so_far

    def end_step(self, tokens_so_far: int) -> float:
        """Returns tokens per second since start_step()."""
        elapsed = time.time() - self._step_start_time + 1e-8
        return (tokens_so_far - self._step_start_tokens) / elapsed

    # ------------------------------------------------------------------ #
    #  Logging                                                             #
    # ------------------------------------------------------------------ #

    def log_step(
        self,
        step: int,
        train_loss: float,
        learning_rate: float,
        global_grad_norm: float,
        tokens_per_sec: float,
        eval_loss: float = None,
        mean_grad_ratio: float = None,
        min_grad_ratio: float = None,
        max_grad_ratio: float = None,
        mean_weight_norm: float = None,
        gpu_memory_gb: float = None,
    ):
        record = StepRecord(
            step=step,
            timestamp=time.time(),
            train_loss=train_loss,
            learning_rate=learning_rate,
            global_grad_norm=global_grad_norm,
            tokens_per_sec=tokens_per_sec,
            eval_loss=eval_loss,
            mean_grad_ratio=mean_grad_ratio,
            min_grad_ratio=min_grad_ratio,
            max_grad_ratio=max_grad_ratio,
            mean_weight_norm=mean_weight_norm,
            gpu_memory_gb=gpu_memory_gb,
        )
        self._records.append(record)

        # Write to JSONL
        self._log_file.write(json.dumps(asdict(record)) + '\n')
        self._log_file.flush()   # flush every step — don't lose data on crash

        # Write to TensorBoard
        if self._tb_writer is not None:
            self._tb_writer.add_scalar('loss/train',        train_loss,       step)
            self._tb_writer.add_scalar('optim/lr',          learning_rate,    step)
            self._tb_writer.add_scalar('grad/global_norm',  global_grad_norm, step)
            self._tb_writer.add_scalar('perf/tokens_per_sec', tokens_per_sec, step)
            if eval_loss is not None:
                self._tb_writer.add_scalar('loss/eval', eval_loss, step)
            if mean_grad_ratio is not None:
                self._tb_writer.add_scalar('grad/mean_ratio',  mean_grad_ratio, step)
                self._tb_writer.add_scalar('grad/min_ratio',   min_grad_ratio,  step)
                self._tb_writer.add_scalar('grad/max_ratio',   max_grad_ratio,  step)
            if mean_weight_norm is not None:
                self._tb_writer.add_scalar('weights/mean_norm', mean_weight_norm, step)
            if gpu_memory_gb is not None:
                self._tb_writer.add_scalar('perf/gpu_memory_gb', gpu_memory_gb, step)

    def log_weight_histograms(self, model, step: int):
        """
        Log weight and gradient histograms to TensorBoard.
        Expensive — call every ~500 steps, not every step.
        """
        if self._tb_writer is None:
            return
        for name, param in model.named_parameters():
            if param.requires_grad:
                self._tb_writer.add_histogram(
                    f'weights/{name}', param.data, step
                )
                if param.grad is not None:
                    self._tb_writer.add_histogram(
                        f'grads/{name}', param.grad, step
                    )

    def close(self):
        self._log_file.close()
        if self._tb_writer is not None:
            self._tb_writer.close()

    # ------------------------------------------------------------------ #
    #  Loading                                                             #
    # ------------------------------------------------------------------ #

    @staticmethod
    def load_records(log_path: str) -> list[dict]:
        """Load a JSONL log file into a list of dicts."""
        records = []
        with open(log_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return records

The JSONL format is append-only and crash-safe — if your job dies at step 8,432, you have records for steps 0–8,431. When you restart, pass `run_dir` to the same logger and it appends to the existing file. You can load and analyze it at any point:

In [ ]:
records = TrainingLogger.load_records('runs/my_experiment/metrics.jsonl')
import pandas as pd
df = pd.DataFrame(records)
print(df[['step', 'train_loss', 'eval_loss', 'global_grad_norm']].tail(20))

---

## 3. The Live Matplotlib Dashboard

For interactive debugging during a run, you want something that updates in real time in the terminal — no browser, no server, no external dependencies beyond Matplotlib.

`plt.ion()` enables interactive mode: `plt.pause(interval)` updates the display and processes GUI events without blocking the training loop. The trick is to update the data on existing plot objects rather than redrawing from scratch — redrawing is slow; updating data is fast.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from collections import deque

class LiveDashboard:
    """
    A real-time training dashboard using Matplotlib interactive mode.
    Updates every `update_every` steps without blocking the training loop.

    Layout (2×3 grid):
        [Train Loss]  [Eval Loss]       [Learning Rate]
        [Grad Norm]   [Grad/Weight ρ]   [Tokens/sec]
    """

    def __init__(
        self,
        update_every: int = 10,
        window: int = 500,         # number of steps to show in rolling window
        smoothing: float = 0.95,   # EMA smoothing for loss curves
    ):
        self.update_every = update_every
        self.window       = window
        self.smoothing    = smoothing

        # Rolling buffers — only keep the last `window` steps
        self._steps      = deque(maxlen=window)
        self._train_loss = deque(maxlen=window)
        self._train_loss_smooth = deque(maxlen=window)
        self._eval_steps = deque(maxlen=window)
        self._eval_loss  = deque(maxlen=window)
        self._grad_norm  = deque(maxlen=window)
        self._grad_ratio = deque(maxlen=window)
        self._lr         = deque(maxlen=window)
        self._tps        = deque(maxlen=window)
        self._ema        = None   # EMA state for loss smoothing

        self._setup_figure()

    def _setup_figure(self):
        plt.ion()
        self.fig = plt.figure(figsize=(15, 8))
        self.fig.suptitle('Training Dashboard', fontsize=13, fontweight='bold')
        gs = gridspec.GridSpec(2, 3, figure=self.fig, hspace=0.4, wspace=0.35)

        # Create axes
        self.ax_loss  = self.fig.add_subplot(gs[0, 0])
        self.ax_eval  = self.fig.add_subplot(gs[0, 1])
        self.ax_lr    = self.fig.add_subplot(gs[0, 2])
        self.ax_gnorm = self.fig.add_subplot(gs[1, 0])
        self.ax_ratio = self.fig.add_subplot(gs[1, 1])
        self.ax_tps   = self.fig.add_subplot(gs[1, 2])

        # Initialize empty line objects — we update .set_data() later
        # This is faster than clearing and replotting
        self.line_loss_raw,    = self.ax_loss.plot([], [], color='#aaaaaa', lw=0.8, alpha=0.5)
        self.line_loss_smooth, = self.ax_loss.plot([], [], color='#2196F3', lw=2.0, label='EMA')
        self.line_eval,        = self.ax_eval.plot([], [], 'o-', color='#FF5722', lw=1.5, ms=4)
        self.line_lr,          = self.ax_lr.plot([], [],   color='#9C27B0', lw=1.5)
        self.line_gnorm,       = self.ax_gnorm.plot([], [], color='#F44336', lw=1.2)
        self.line_ratio,       = self.ax_ratio.plot([], [], color='#4CAF50', lw=1.5)
        self.line_tps,         = self.ax_tps.plot([], [],  color='#FF9800', lw=1.2)

        # Healthy ratio band
        self.ax_ratio.axhspan(1e-3, 1e-2, alpha=0.15, color='green', label='healthy')

        # Labels
        self.ax_loss.set(title='Train Loss', ylabel='loss')
        self.ax_eval.set(title='Eval Loss',  ylabel='loss')
        self.ax_lr.set(  title='Learning Rate', ylabel='lr')
        self.ax_gnorm.set(title='Global Grad Norm', ylabel='norm')
        self.ax_ratio.set(title='Grad/Weight Ratio ρ', ylabel='ρ')
        self.ax_tps.set(  title='Throughput', ylabel='tok/s')

        for ax in [self.ax_gnorm, self.ax_ratio]:
            ax.set_yscale('log')

        self.ax_loss.legend(fontsize=8)
        self.ax_ratio.legend(fontsize=8)

        plt.show(block=False)
        plt.pause(0.01)

    def _ema_update(self, value: float) -> float:
        """Exponential moving average for loss smoothing."""
        if self._ema is None:
            self._ema = value
        else:
            self._ema = self.smoothing * self._ema + (1 - self.smoothing) * value
        return self._ema

    def update(
        self,
        step: int,
        train_loss: float,
        learning_rate: float,
        global_grad_norm: float,
        tokens_per_sec: float,
        eval_loss: float = None,
        mean_grad_ratio: float = None,
    ):
        # Always append to buffers
        self._steps.append(step)
        self._train_loss.append(train_loss)
        self._train_loss_smooth.append(self._ema_update(train_loss))
        self._lr.append(learning_rate)
        self._grad_norm.append(global_grad_norm)
        self._tps.append(tokens_per_sec)

        if eval_loss is not None:
            self._eval_steps.append(step)
            self._eval_loss.append(eval_loss)

        if mean_grad_ratio is not None:
            self._grad_ratio.append(mean_grad_ratio)

        # Only redraw every `update_every` steps
        if step % self.update_every != 0:
            return

        steps = list(self._steps)

        def _update_line(line, xs, ys):
            if xs and ys:
                line.set_data(xs, ys)
                line.axes.relim()
                line.axes.autoscale_view()

        _update_line(self.line_loss_raw,    steps, list(self._train_loss))
        _update_line(self.line_loss_smooth, steps, list(self._train_loss_smooth))
        _update_line(self.line_lr,          steps, list(self._lr))
        _update_line(self.line_gnorm,       steps, list(self._grad_norm))
        _update_line(self.line_tps,         steps, list(self._tps))

        if self._eval_loss:
            _update_line(self.line_eval, list(self._eval_steps), list(self._eval_loss))

        if self._grad_ratio:
            _update_line(self.line_ratio, steps[-len(self._grad_ratio):],
                         list(self._grad_ratio))

        # Add step count to title
        self.fig.suptitle(
            f'Training Dashboard — step {step:,}  '
            f'loss={train_loss:.4f}  '
            f'lr={learning_rate:.2e}',
            fontsize=13, fontweight='bold'
        )

        self.fig.canvas.draw_idle()
        plt.pause(0.01)   # process GUI events — keeps window responsive

    def save(self, path: str):
        """Save the current dashboard state as a PNG."""
        self.fig.savefig(path, dpi=150, bbox_inches='tight')
        print(f"Dashboard saved to {path}")

    def close(self):
        plt.ioff()
        plt.close(self.fig)

**The `plt.pause(0.01)` contract.** This call has two jobs: it flushes pending draw commands and it processes GUI events (mouse, keyboard, resize). Without it the window freezes[^pause]. The 0.01 second duration is a lower bound

[^pause]: Matplotlib GUI backends (Tk, Qt, Wx) run an OS event loop to handle window resize, mouse clicks, and keyboard input. `plt.pause()` temporarily yields control to that loop. Without it the OS marks the window as unresponsive and may grey it out. — if your training step takes 0.5 seconds, the pause is negligible. If a step takes 1ms (tiny model, fast GPU), you are spending 1% of training time on the dashboard. Increase `update_every` if throughput matters.

**Why `set_data()` instead of `ax.clear()` + `ax.plot()`?** Clearing and replotting rebuilds all the Matplotlib artists from scratch — slow and causes visual flickering. Calling `set_data()` on existing `Line2D` objects updates only the data, then `relim()` + `autoscale_view()` adjusts the axis bounds. [This is roughly 10× faster and flicker-free.]{.mark}

---

## 4. TensorBoard Integration

TensorBoard's strength over the live dashboard is comparison across runs. You can overlay the loss curves of 10 different hyperparameter configurations in a single view, zoom into any time range, and inspect weight histograms at any checkpoint.

The `TrainingLogger` above already handles TensorBoard writes. Here is what to log and when:

In [ ]:
# In your training loop — full example with logger + dashboard + TensorBoard:

import torch
import torch.nn as nn

def compute_grad_stats(model):
    """Compute global grad norm and per-layer ratio stats in one pass."""
    total_sq   = 0.0
    ratios     = []
    norms      = []

    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        if module.weight.grad is None:
            continue
        g = module.weight.grad.norm().item()
        w = module.weight.norm().item()
        total_sq += g ** 2
        ratios.append(g / (w + 1e-8))
        norms.append(w)

    global_norm = total_sq ** 0.5
    return {
        'global_grad_norm':  global_norm,
        'mean_grad_ratio':   float(np.mean(ratios))   if ratios else 0.0,
        'min_grad_ratio':    float(np.min(ratios))    if ratios else 0.0,
        'max_grad_ratio':    float(np.max(ratios))    if ratios else 0.0,
        'mean_weight_norm':  float(np.mean(norms))    if norms  else 0.0,
    }


def get_gpu_memory_gb() -> float:
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e9
    return 0.0


def train(model, optimizer, scheduler, dataloader, config, run_dir):
    logger    = TrainingLogger(run_dir, use_tensorboard=True)
    dashboard = LiveDashboard(update_every=10, window=300)

    total_tokens = 0
    step         = 0

    for x, y in dataloader:
        if step >= config.max_steps:
            break

        # --- Throughput timing ---
        logger.start_step(total_tokens)

        # --- Forward + backward ---
        x, y = x.to(config.device), y.to(config.device)
        _, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()

        # --- Gradient stats (before clipping) ---
        stats = compute_grad_stats(model)

        # --- Clip ---
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # --- Update ---
        optimizer.step()
        scheduler.step()

        # --- Metrics ---
        total_tokens += x.numel()
        tps = logger.end_step(total_tokens)
        lr  = scheduler.get_last_lr()[0]

        # Log every step
        logger.log_step(
            step=step,
            train_loss=loss.item(),
            learning_rate=lr,
            global_grad_norm=stats['global_grad_norm'],
            tokens_per_sec=tps,
            mean_grad_ratio=stats['mean_grad_ratio'],
            min_grad_ratio=stats['min_grad_ratio'],
            max_grad_ratio=stats['max_grad_ratio'],
            mean_weight_norm=stats['mean_weight_norm'],
            gpu_memory_gb=get_gpu_memory_gb(),
        )

        # Update live dashboard
        dashboard.update(
            step=step,
            train_loss=loss.item(),
            learning_rate=lr,
            global_grad_norm=stats['global_grad_norm'],
            tokens_per_sec=tps,
            mean_grad_ratio=stats['mean_grad_ratio'],
        )

        # Expensive logging — every 500 steps
        if step % 500 == 0:
            eval_loss = evaluate(model, eval_dataloader, config.device)
            logger.log_step(step=step, train_loss=loss.item(),
                            learning_rate=lr,
                            global_grad_norm=stats['global_grad_norm'],
                            tokens_per_sec=tps, eval_loss=eval_loss)
            logger.log_weight_histograms(model, step)
            dashboard.update(step=step, train_loss=loss.item(),
                             learning_rate=lr,
                             global_grad_norm=stats['global_grad_norm'],
                             tokens_per_sec=tps, eval_loss=eval_loss)
            dashboard.save(f'{run_dir}/dashboard_step{step}.png')

        step += 1

    logger.close()
    dashboard.close()

### TensorBoard: What Each View Tells You

**Scalars tab** — where you spend most of your time. The key panels:

- `loss/train` and `loss/eval` on the same chart: the gap between them is your overfitting signal. A widening gap after a certain step means you've hit the compute-optimal point for this dataset size — more training steps help train loss but hurt generalization.
- `grad/global_norm`: look for spikes. [A spike at step $k$ that is followed by a loss spike at step $k+1$ or $k+2$ is a confirmed instability]{.mark} event. [The gradient norm spike is the *cause*; the loss spike is the *effect*.]{.underline} You need to see both to be sure.
- `grad/mean_ratio` with `grad/min_ratio` and `grad/max_ratio`: the spread between min and max ratio tells you how uneven gradient flow is across layers. A large spread means some layers are training much faster than others — pathological.

**Histograms tab** — logged every 500 steps. The weight histogram for a healthy layer should be roughly Gaussian and should slowly broaden over training as weights move away from their initialization. A histogram that stays narrow means that layer is not learning. A histogram with a bimodal shape (weights splitting into two groups) is unusual and worth investigating.

**The "Runs" selector** — TensorBoard's killer feature. Run two experiments (e.g., with and without LN) with different `run_name` values, point TensorBoard at the parent `run_dir`, and overlay their curves. This is how you run controlled ablations.

---

## 5. The Sawtooth Chart

This deserves its own section because it appears constantly and means different things depending on context.

A **sawtooth pattern** on any metric is a repeated spike-and-drop. The key is identifying what is spiking and what is dropping:

### Sawtooth on train loss — normal

This is the most common and least alarming sawtooth. It appears when you log train loss at every step on a small dataset with gradient accumulation. Within each accumulation cycle, the loss decreases slightly (you've just updated). At the start of the next cycle, you hit a new batch — slightly different distribution, slightly higher loss. The result is a micro-sawtooth at the accumulation frequency.

If you're not using gradient accumulation, a mild step-to-step sawtooth is just stochastic batch noise. Smooth it with EMA (the `smoothing=0.95` parameter in `LiveDashboard`) and move on.

### Sawtooth on gradient norm — investigate

A periodic spike in gradient norm that repeats at a fixed frequency is suspicious. Common causes:

- **Data ordering artifact**: if your dataset has periodic structure (e.g., every 100th batch is a different domain), the gradient norm will spike on those batches. Fix: shuffle your dataset more aggressively.
- **LR schedule phase transition**: at the boundary between warmup and decay, the LR changes sharply — the gradient norm often spikes briefly. Expected and harmless.
- **Checkpoint loading**: if you load a checkpoint mid-training and the optimizer state does not match (e.g., you changed batch size), the first few steps after loading can produce anomalous gradient norms. Always save and load the full optimizer state.

### Sawtooth on GPU memory — memory leak

If GPU memory grows monotonically across steps and then drops back (or worse, grows without bound until OOM), you have a memory leak. The most common cause in training loops:

```python
# WRONG — keeps the computation graph alive across steps
losses.append(loss)

# CORRECT — detach before storing
losses.append(loss.item())
```

A subtler version: storing model outputs for later analysis without detaching:

```python
# WRONG
all_logits.append(model(x))    # keeps entire graph alive

# CORRECT
with torch.no_grad():
    all_logits.append(model(x).cpu())
```

Monitor GPU memory with `torch.cuda.memory_allocated()` every 50 steps. A truly flat line is correct. [[A slowly rising line means you have a leak]{.underline} — *binary search* your training loop to find it.

### Sawtooth on the gradient-to-weight ratio — compaction / context reset

In longer training runs, if you are periodically resetting optimizer state (e.g., restarting training, reloading from checkpoint without optimizer state), the ratio will spike because the optimizer's second-moment estimate (Adam's $v_t$) has been reset to zero. The first few steps after reset have anomalously large effective updates. This is the "compaction sawtooth" analogy from the multi-agent series — a periodic reset producing a periodic pattern.

---

## 6. Smoothing and Noise

Raw step-level loss is almost always too noisy to read — batch variance dominates. The right smoothing method depends on what you are looking for:

**Exponential Moving Average (EMA):** Best for the live dashboard. Updates online with O(1) memory. The `smoothing` parameter is the decay factor — 0.9 gives a short window (~10 steps), 0.99 gives a long window (~100 steps). Use 0.95 as the default.

In [ ]:
def ema(values: list[float], alpha: float = 0.05) -> list[float]:
    """alpha = 1 - smoothing. Lower alpha = smoother."""
    smoothed, s = [], values[0]
    for v in values:
        s = (1 - alpha) * s + alpha * v
        smoothed.append(s)
    return smoothed

**Simple moving average (SMA):** Better for post-hoc analysis. Use a window of 50–200 steps. Does not distort the timing of events the way EMA does (EMA lags; SMA has a flat lag equal to half the window).

```python
import pandas as pd

df['loss_sma50'] = df['train_loss'].rolling(window=50, min_periods=1).mean()
```

**Log scale for loss:** When loss spans multiple orders of magnitude (early training vs late), plot on a log scale. A straight line on a log-loss vs linear-step plot means exponential convergence — the best you can hope for in the early phase.

---

## 7. Post-Run Analysis

After the run, the JSONL file is your source of truth. A few analysis patterns worth keeping:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load
df = pd.DataFrame(TrainingLogger.load_records('runs/my_experiment/metrics.jsonl'))
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

# 1. Find the step where eval loss was minimum
best_step = df.dropna(subset=['eval_loss'])['eval_loss'].idxmin()
print(f"Best eval loss at step {df.loc[best_step, 'step']}: "
      f"{df.loc[best_step, 'eval_loss']:.4f}")

# 2. Find gradient norm spikes (> 3 std above rolling mean)
rolling_mean = df['global_grad_norm'].rolling(100).mean()
rolling_std  = df['global_grad_norm'].rolling(100).std()
spikes = df[df['global_grad_norm'] > rolling_mean + 3 * rolling_std]
print(f"\nGradient norm spikes at steps: {spikes['step'].tolist()}")

# 3. Compute average throughput (excluding first 10 steps — warmup)
tps_stable = df[df['step'] > 10]['tokens_per_sec']
print(f"\nMean throughput: {tps_stable.mean():.0f} tok/s "
      f"(std: {tps_stable.std():.0f})")

# 4. Plot loss + grad norm together — aligned x-axis
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.semilogy(df['step'], df['train_loss'].rolling(50).mean(), label='train (sma50)')
eval_df = df.dropna(subset=['eval_loss'])
ax1.semilogy(eval_df['step'], eval_df['eval_loss'], 'o-', ms=4, label='eval')
ax1.set_ylabel('Loss (log)')
ax1.legend()
ax1.set_title('Training Run Analysis')

ax2.semilogy(df['step'], df['global_grad_norm'], alpha=0.4, color='red', label='raw')
ax2.semilogy(df['step'], df['global_grad_norm'].rolling(50).mean(),
             color='darkred', lw=2, label='sma50')
ax2.axhline(1.0, color='gray', linestyle='--', label='clip threshold')
ax2.set_xlabel('Step')
ax2.set_ylabel('Grad Norm (log)')
ax2.legend()

plt.tight_layout()
plt.savefig('run_analysis.png', dpi=150)
plt.show()

---

## 8. Putting It All Together: The Training Loop Template

The complete template — everything connected:

In [ ]:
from tutorial_02 import GPT, NanoGPTConfig
from tutorial_03 import Tokenizer

def make_cosine_scheduler(optimizer, warmup_steps, max_steps, min_lr_ratio=0.1):
    """Linear warmup + cosine decay. Tutorial 9 covers this in full."""
    import math
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(max_steps - warmup_steps, 1)
        cosine   = 0.5 * (1 + math.cos(math.pi * progress))
        return min_lr_ratio + (1 - min_lr_ratio) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def full_training_run(run_name: str = 'nano_gpt_v1'):
    # Config
    config    = NanoGPTConfig()
    run_dir   = f'runs/{run_name}'
    max_steps = 5000
    batch_size = 8
    device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Data
    tok  = Tokenizer.load('nano_tokenizer.json')
    text = open('tinyshakespeare.txt').read()
    data = torch.tensor(tok.encode(text), dtype=torch.long)
    block = config.max_seq_len

    def get_batch(split='train'):
        n     = int(0.9 * len(data))
        d     = data[:n] if split == 'train' else data[n:]
        ix    = torch.randint(len(d) - block, (batch_size,))
        x     = torch.stack([d[i   : i+block  ] for i in ix]).to(device)
        y     = torch.stack([d[i+1 : i+block+1] for i in ix]).to(device)
        return x, y

    @torch.no_grad()
    def evaluate(n_batches=20):
        model.eval()
        losses = [model(*get_batch('val'))[1].item() for _ in range(n_batches)]
        model.train()
        return float(np.mean(losses))

    # Model
    model     = GPT(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
    scheduler = make_cosine_scheduler(optimizer, warmup_steps=100, max_steps=max_steps)

    # Logging
    logger    = TrainingLogger(run_dir, run_name, use_tensorboard=True)
    dashboard = LiveDashboard(update_every=20, window=500)

    total_tokens = 0

    for step in range(max_steps):
        x, y = get_batch('train')
        logger.start_step(total_tokens)

        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()

        stats = compute_grad_stats(model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_tokens += x.numel()
        tps = logger.end_step(total_tokens)
        lr  = scheduler.get_last_lr()[0]

        logger.log_step(
            step=step,
            train_loss=loss.item(),
            learning_rate=lr,
            global_grad_norm=stats['global_grad_norm'],
            tokens_per_sec=tps,
            mean_grad_ratio=stats['mean_grad_ratio'],
            min_grad_ratio=stats['min_grad_ratio'],
            max_grad_ratio=stats['max_grad_ratio'],
            mean_weight_norm=stats['mean_weight_norm'],
            gpu_memory_gb=get_gpu_memory_gb(),
        )

        dashboard.update(
            step=step,
            train_loss=loss.item(),
            learning_rate=lr,
            global_grad_norm=stats['global_grad_norm'],
            tokens_per_sec=tps,
            mean_grad_ratio=stats['mean_grad_ratio'],
        )

        if step % 500 == 0 or step == max_steps - 1:
            el = evaluate()
            print(f"step {step:5d}  train={loss.item():.4f}  eval={el:.4f}  "
                  f"lr={lr:.2e}  grad={stats['global_grad_norm']:.3f}  "
                  f"tps={tps:.0f}")
            logger.log_weight_histograms(model, step)
            dashboard.save(f'{run_dir}/dashboard_{step:05d}.png')

    logger.close()
    dashboard.close()
    print(f"\nRun complete. Metrics at: {run_dir}/metrics.jsonl")
    print(f"TensorBoard: tensorboard --logdir runs/")

if __name__ == '__main__':
    full_training_run('nano_gpt_v1')

---

## Summary

| Concept | Key detail |
|---|---|
| Log to JSONL first | Visualization tools change; `.jsonl` files are forever. Flush every step. |
| `plt.ion()` + `plt.pause()` | Enables non-blocking live updates. Pause processes GUI events — required to keep window responsive. |
| `set_data()` not `ax.clear()` | Update existing `Line2D` objects — 10× faster, no flicker. |
| EMA smoothing | `alpha=0.05` (i.e., `smoothing=0.95`) balances noise reduction vs lag. |
| TensorBoard histograms | Log every 500 steps — expensive. Reveals weight distribution drift. |
| Sawtooth on loss | Batch noise or accumulation cycle — smooth it. |
| Sawtooth on grad norm | Data ordering artifact, LR phase transition, or bad checkpoint load. |
| Sawtooth on GPU memory | Memory leak — `.item()` every stored loss, `.cpu()` every stored tensor. |
| Sawtooth on ratio | Optimizer state reset — always save and load full optimizer state with checkpoints. |
| Pre-clip vs post-clip norm | Always compute and log the pre-clip norm — it tells you how often clipping fires. |
| `loss/train` vs `loss/eval` gap | Widening gap = overfitting. Use to identify compute-optimal stopping point. |
| Spike → loss spike lag | Gradient norm spike causes loss spike 1–2 steps later. Seeing both confirms instability. |

---

## Exercises

**1.** Run `full_training_run` and deliberately introduce the memory leak: change `loss.item()` to `loss` in the logger call. Monitor `gpu_memory_gb` over 500 steps and confirm it grows. Fix the leak and confirm the memory line becomes flat.

**2.** Add a seventh panel to `LiveDashboard` showing GPU memory over time. Use `torch.cuda.memory_allocated() / 1e9`. Handle the case where CUDA is unavailable gracefully (show 0 or hide the panel).

**3.** Implement a `compare_runs` function that takes a list of JSONL paths and a list of run names, loads all of them into DataFrames, and produces a single plot with all train loss curves (SMA-smoothed) overlaid on one axis and all eval loss curves on another. Use this to compare `lr=1e-4` vs `lr=3e-4` vs `lr=1e-3`.

**4.** The `compute_grad_stats` function iterates over all named modules twice (once for grad norm, once implicitly via the model). Refactor it to make a single pass, collecting all statistics simultaneously. Verify the output is identical.

**5.** Add anomaly detection to `TrainingLogger.log_step()`: if `global_grad_norm` is more than 5× the rolling mean of the last 100 steps, write a warning line to a separate `alerts.jsonl` file. After a training run, load `alerts.jsonl` and cross-reference the alert steps with the loss curve.

**6.** TensorBoard's `add_histogram` is called with raw parameter tensors. Modify `log_weight_histograms` to also log the ratio of gradient histogram spread to weight histogram spread — `grad.std() / weight.std()` — as a scalar. Verify this correlates with the `grad_weight_ratio` from `compute_grad_stats`.